In [1]:
import copy
import json
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


SEED = 42
DATASET_NAME = "CIFAR10"
BATCH_SIZE = 512
NUM_WORKERS = 0
DATA_ROOT = "./data"
INPUT_DIM = 32 * 32 * 3
NUM_CLASSES = 10

MAX_EPOCHS_A = 8
MAX_EPOCHS_E4 = 15
PATIENCE_E4 = 4

E1_LR = 1e-3
E2_LR = 1e-3
E3_LR = 1e-3
E4_LR = 1e-3

O1_LR = 1e-1
O2_LR = 1e-5
O3_LR = 1e-2
O3_MOMENTUM = 0.9
O3_WEIGHT_DECAY = 1e-4


@dataclass
class ModelConfig:
    hidden_sizes: tuple[int, ...]
    dropout: float
    batchnorm: bool
    activation: str = "ReLU"


class MLP(nn.Module):
    def __init__(self, input_dim: int, num_classes: int, config: ModelConfig):
        super().__init__()
        layers: list[nn.Module] = [nn.Flatten()]
        in_dim = input_dim

        for hidden_dim in config.hidden_sizes:
            layers.append(nn.Linear(in_dim, hidden_dim))
            if config.batchnorm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            if config.dropout > 0:
                layers.append(nn.Dropout(config.dropout))
            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    correct = (preds == targets).sum().item()
    return correct / targets.size(0)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        total_samples += batch_size
        total_loss += loss.item() * batch_size
        total_correct += (torch.argmax(logits, dim=1) == y).sum().item()

    return total_loss / total_samples, total_correct / total_samples


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            batch_size = y.size(0)
            total_samples += batch_size
            total_loss += loss.item() * batch_size
            total_correct += (torch.argmax(logits, dim=1) == y).sum().item()

    return total_loss / total_samples, total_correct / total_samples


def run_experiment(
    experiment_id: str,
    model_config: ModelConfig,
    optimizer_name: str,
    lr: float,
    max_epochs: int,
    train_loader,
    val_loader,
    device,
    momentum: float = 0.0,
    weight_decay: float = 0.0,
    early_stopping: bool = False,
    patience: int = 4,
):
    model = MLP(input_dim=INPUT_DIM, num_classes=NUM_CLASSES, config=model_config).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay,
        )
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for _epoch in range(1, max_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if val_loss < best_val_loss:
            best_val_loss = val_loss

        if early_stopping and epochs_without_improvement >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    result = {
        "experiment_id": experiment_id,
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": (
            f"hidden={list(model_config.hidden_sizes)}, act={model_config.activation}, "
            f"dropout={model_config.dropout}, batchnorm={model_config.batchnorm}"
        ),
        "optimizer": optimizer_name,
        "lr": lr,
        "momentum": momentum if optimizer_name == "SGD" else 0.0,
        "weight_decay": weight_decay,
        "epochs_trained": len(history["train_loss"]),
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
    }

    return model, history, result


def plot_best_curves(history: dict, save_path: Path) -> None:
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(epochs, history["train_loss"], label="train_loss")
    axes[0].plot(epochs, history["val_loss"], label="val_loss")
    axes[0].set_title("E4: Loss Curves")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history["train_acc"], label="train_acc")
    axes[1].plot(epochs, history["val_acc"], label="val_acc")
    axes[1].set_title("E4: Accuracy Curves")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(save_path, dpi=140)
    plt.close(fig)


def plot_lr_extremes(o1_history: dict, o2_history: dict, save_path: Path) -> None:
    e1 = np.arange(1, len(o1_history["train_loss"]) + 1)
    e2 = np.arange(1, len(o2_history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(e1, o1_history["val_loss"], marker="o", label="O1 val_loss (lr=1e-1)")
    axes[0].plot(e2, o2_history["val_loss"], marker="o", label="O2 val_loss (lr=1e-5)")
    axes[0].set_title("Validation Loss: LR Extremes")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(e1, o1_history["val_acc"], marker="o", label="O1 val_acc (lr=1e-1)")
    axes[1].plot(e2, o2_history["val_acc"], marker="o", label="O2 val_acc (lr=1e-5)")
    axes[1].set_title("Validation Accuracy: LR Extremes")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(save_path, dpi=140)
    plt.close(fig)


def main():
    set_seed(SEED)

    if "__file__" in globals():
        base_dir = Path(__file__).resolve().parent
    else:
        base_dir = Path.cwd()
    artifacts_dir = base_dir / "artifacts"
    figures_dir = artifacts_dir / "figures"
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ]
    )
    train_full = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=transform)

    val_size = int(0.2 * len(train_full))
    train_size = len(train_full) - val_size
    train_dataset, val_dataset = random_split(
        train_full,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    config_e1 = ModelConfig(hidden_sizes=(256, 128), dropout=0.0, batchnorm=False)
    config_e2 = ModelConfig(hidden_sizes=(256, 128), dropout=0.3, batchnorm=False)
    config_e3 = ModelConfig(hidden_sizes=(256, 128), dropout=0.0, batchnorm=True)

    all_results = []
    histories = {}

    _m1, h1, r1 = run_experiment(
        "E1",
        model_config=config_e1,
        optimizer_name="Adam",
        lr=E1_LR,
        max_epochs=MAX_EPOCHS_A,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
    )
    all_results.append(r1)
    histories["E1"] = h1

    _m2, h2, r2 = run_experiment(
        "E2",
        model_config=config_e2,
        optimizer_name="Adam",
        lr=E2_LR,
        max_epochs=MAX_EPOCHS_A,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
    )
    all_results.append(r2)
    histories["E2"] = h2

    _m3, h3, r3 = run_experiment(
        "E3",
        model_config=config_e3,
        optimizer_name="Adam",
        lr=E3_LR,
        max_epochs=MAX_EPOCHS_A,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
    )
    all_results.append(r3)
    histories["E3"] = h3

    better_source = "E2" if r2["best_val_accuracy"] >= r3["best_val_accuracy"] else "E3"
    best_reg_config = config_e2 if better_source == "E2" else config_e3

    best_model, h4, r4 = run_experiment(
        "E4",
        model_config=best_reg_config,
        optimizer_name="Adam",
        lr=E4_LR,
        max_epochs=MAX_EPOCHS_E4,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        early_stopping=True,
        patience=PATIENCE_E4,
    )
    all_results.append(r4)
    histories["E4"] = h4

    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)

    torch.save(best_model.state_dict(), artifacts_dir / "best_model.pt")

    best_config_payload = {
        "dataset": DATASET_NAME,
        "seed": SEED,
        "device_used": device.type,
        "source_regularization_experiment": better_source,
        "experiment_id": "E4",
        "architecture": {
            "hidden_sizes": list(best_reg_config.hidden_sizes),
            "dropout": best_reg_config.dropout,
            "batchnorm": best_reg_config.batchnorm,
            "activation": best_reg_config.activation,
        },
        "training": {
            "optimizer": "Adam",
            "lr": E4_LR,
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS_E4,
            "early_stopping_patience": PATIENCE_E4,
            "weight_decay": 0.0,
        },
        "metrics": {
            "best_val_accuracy": r4["best_val_accuracy"],
            "best_val_loss": r4["best_val_loss"],
            "test_accuracy": test_acc,
            "test_loss": test_loss,
        },
    }

    with (artifacts_dir / "best_config.json").open("w", encoding="utf-8") as f:
        json.dump(best_config_payload, f, ensure_ascii=False, indent=2)

    _, h_o1, r_o1 = run_experiment(
        "O1",
        model_config=best_reg_config,
        optimizer_name="Adam",
        lr=O1_LR,
        max_epochs=5,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
    )
    all_results.append(r_o1)
    histories["O1"] = h_o1

    _, h_o2, r_o2 = run_experiment(
        "O2",
        model_config=best_reg_config,
        optimizer_name="Adam",
        lr=O2_LR,
        max_epochs=5,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
    )
    all_results.append(r_o2)
    histories["O2"] = h_o2

    _, h_o3, r_o3 = run_experiment(
        "O3",
        model_config=best_reg_config,
        optimizer_name="SGD",
        lr=O3_LR,
        max_epochs=10,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        momentum=O3_MOMENTUM,
        weight_decay=O3_WEIGHT_DECAY,
    )
    all_results.append(r_o3)
    histories["O3"] = h_o3

    runs = pd.DataFrame(all_results)
    runs["best_val_accuracy"] = runs["best_val_accuracy"].map(lambda x: round(float(x), 6))
    runs["best_val_loss"] = runs["best_val_loss"].map(lambda x: round(float(x), 6))
    runs.to_csv(artifacts_dir / "runs.csv", index=False)

    plot_best_curves(h4, figures_dir / "curves_best.png")
    plot_lr_extremes(h_o1, h_o2, figures_dir / "curves_lr_extremes.png")

    summary = {
        "device": device.type,
        "train_size": train_size,
        "val_size": val_size,
        "test_size": len(test_dataset),
        "test_accuracy": test_acc,
        "test_loss": test_loss,
        "best_source": better_source,
    }
    with (artifacts_dir / "summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("Completed HW08-09 experiments")
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


100.0%
c:\Users\Beliy\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Completed HW08-09 experiments
{
  "device": "cpu",
  "train_size": 40000,
  "val_size": 10000,
  "test_size": 10000,
  "test_accuracy": 0.5315,
  "test_loss": 1.351429920578003,
  "best_source": "E3"
}
